# Ch02-02: 연속 시간 마르코프 체인과 포아송 과정 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

Thinning 알고리즘으로 비동질 포아송 과정을 시뮬레이션합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def thinning_poisson(lambda_func, lambda_max, T, seed=42):
    np.random.seed(seed)
    events = []
    t = 0
    while t < T:
        t += np.random.exponential(1 / lambda_max)
        if t < T and np.random.rand() < lambda_func(t) / lambda_max:
            events.append(t)
    return np.array(events)

lambda_func = lambda t: 5 + 3 * np.sin(2 * np.pi * t / 24)
events = thinning_poisson(lambda_func, lambda_max=8, T=72)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
t_grid = np.linspace(0, 72, 1000)
axes[0].plot(t_grid, lambda_func(t_grid), 'b-')
axes[0].set_ylabel('λ(t)')
axes[0].set_title('강도 함수')

# 시간대별 도착 빈도
axes[1].hist(events, bins=72, edgecolor='black', alpha=0.7, density=True)
axes[1].plot(t_grid, lambda_func(t_grid) / np.trapz(lambda_func(t_grid), t_grid), 'r-')
axes[1].set_xlabel('시간')
axes[1].set_ylabel('상대 빈도')
axes[1].set_title(f'도착 분포 (총 {len(events)}건)')
plt.tight_layout()
plt.show()

**결과 해석**: Thinning은 강도 함수의 상한 λ_max로 후보를 생성한 후 λ(t)/λ_max 확률로 채택합니다. 효율은 E[λ(t)]/λ_max에 비례합니다.

---
## 문제 2 풀이

이산 사건 시뮬레이션으로 M/M/1 큐를 구현합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_mm1(lam, mu, n_customers=10000):
    np.random.seed(42)
    inter_arrivals = np.random.exponential(1/lam, n_customers)
    service_times = np.random.exponential(1/mu, n_customers)
    arrivals = np.cumsum(inter_arrivals)

    departures = np.zeros(n_customers)
    departures[0] = arrivals[0] + service_times[0]
    for i in range(1, n_customers):
        start = max(arrivals[i], departures[i-1])
        departures[i] = start + service_times[i]

    waits = departures - arrivals - service_times
    system_times = departures - arrivals
    return {'W': system_times.mean(), 'Wq': waits.mean(),
            'L': lam * system_times.mean(), 'Lq': lam * waits.mean()}

rhos = np.arange(0.1, 0.99, 0.1)
L_sim, L_theory = [], []
for rho in rhos:
    lam, mu = rho, 1.0
    result = simulate_mm1(lam, mu)
    L_sim.append(result['L'])
    L_theory.append(rho / (1 - rho))

plt.figure(figsize=(10, 6))
plt.plot(rhos, L_theory, 'b-o', label='이론 L=ρ/(1-ρ)')
plt.plot(rhos, L_sim, 'r--s', label='시뮬레이션')
plt.xlabel('이용률 ρ')
plt.ylabel('평균 시스템 내 고객 수 L')
plt.title('M/M/1 큐: 이론 vs 시뮬레이션')
plt.legend()
plt.yscale('log')
plt.show()

**결과 해석**: ρ→1이면 L이 급격히 증가합니다. Little의 법칙 L=λW가 시뮬레이션에서도 성립함을 확인합니다.

---
## 문제 3 풀이

Gillespie 알고리즘으로 확률적 SIR을 구현합니다.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

def gillespie_sir(S0, I0, R0, beta, gamma, T):
    S, I, R = S0, I0, R0
    N = S + I + R
    t = 0
    trajectory = [(t, S, I, R)]

    while t < T and I > 0:
        rate_infect = beta * S * I / N
        rate_recover = gamma * I
        total_rate = rate_infect + rate_recover

        if total_rate == 0:
            break
        dt = np.random.exponential(1 / total_rate)
        t += dt

        if np.random.rand() < rate_infect / total_rate:
            S -= 1; I += 1
        else:
            I -= 1; R += 1
        trajectory.append((t, S, I, R))

    return np.array(trajectory)

def sir_ode(y, t, beta, gamma, N):
    S, I, R = y
    return [-beta*S*I/N, beta*S*I/N - gamma*I, gamma*I]

N, beta, gamma = 1000, 0.3, 0.1
np.random.seed(42)

fig, ax = plt.subplots(figsize=(12, 6))
for _ in range(20):
    traj = gillespie_sir(N-10, 10, 0, beta, gamma, 200)
    ax.plot(traj[:,0], traj[:,2]/N, 'r-', alpha=0.15)

t_ode = np.linspace(0, 200, 1000)
sol = odeint(sir_ode, [N-10, 10, 0], t_ode, args=(beta, gamma, N))
ax.plot(t_ode, sol[:,1]/N, 'k-', linewidth=3, label='결정론적 ODE')
ax.set_xlabel('시간')
ax.set_ylabel('감염 비율 I/N')
ax.set_title('SIR 모형: Gillespie vs ODE')
ax.legend()
plt.show()

**결과 해석**: 확률적 모형은 개체 수가 적을 때 결정론적 모형과 큰 차이를 보입니다. 멸종(I=0)이 일어날 수 있어 일부 궤적은 조기 종료됩니다.

---
## 확장 토론

CTMC와 포아송 과정은 큐잉, 신뢰성 공학, 생물학적 과정 모델링의 기초입니다. Gillespie 알고리즘은 화학 반응 시뮬레이션에서 시작했지만, 전염병 모형 등 다양한 분야에 적용됩니다.